# 4. Train
1. train set 준비
2. cluster center 초기화
3. NetVLAD 파라미터 초기화
4. Dataset 구성 (Query 기준)
5. triplet mining
6. loss 계산
7. optimizer step
8. validation


In [46]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
from collections import namedtuple
import random

In [47]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
import torch.autograd as Variable


import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


In [48]:
import sklearn
from sklearn.neighbors import NearestNeighbors

In [49]:
root_dir = './data/Pittsburgh250k/'
struct_dir = os.path.join(root_dir, 'netvlad_v100_datasets/datasets/')
queries_dir = os.path.join(root_dir, 'queries_real/')

In [50]:
def parse_dbStruct(structfile, dbPath):

    structfile
    dataset = structfile  #db의 이름을 넣기 위한 위치

    mat = loadmat(os.path.join(dbPath,structfile))

    matStruct = mat['dbStruct'].item()

    #debugging 용 출력
    print(len(matStruct))
    first_col = list(map(lambda x: x[0], matStruct))
    for i in range(len(matStruct)):
        print(f"matStruct[{i}] :{first_col[i]}")

    whichSet = matStruct[0].item()

    dbImage = [f[0].item() for f in matStruct[1]]  #이미지리스트
    utmDb = matStruct[2].T

    qImage = [f[0].item() for f in matStruct[3]] #쿼리 이미지
    utmQ = matStruct[4].T

    numDb = matStruct[5].item()
    numQ = matStruct[6].item()

    posDistThr = matStruct[7].item()  #25
    posDistSqThr = matStruct[8].item() #625 --> 25^2
    nonTrivPosDistSqThr = matStruct[9].item() #100 -->10^2

    return dbStruct(whichSet, dataset, dbImage, utmDb, qImage, 
        utmQ, numDb, numQ, posDistThr, 
        posDistSqThr, nonTrivPosDistSqThr)

dbStruct = namedtuple('dbStruct', ['whichSet', 'dataset', 
    'dbImage', 'utmDb', 'qImage', 'utmQ', 'numDb', 'numQ',
    'posDistThr', 'posDistSqThr', 'nonTrivPosDistSqThr'])

train = parse_dbStruct('pitts30k_train.mat',struct_dir)

def input_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
    ])

class WholeDatset(data.Dataset):

    def __init__(self, dbPath, stPath, qPath, structFile, transform=None, onlyDB=False):
        super().__init__()  #parent class 초기화용이나, 현재는 크게필요하지 않음. 
        self.input_transform = transform #tensor로 변환
        self.dbStruct = parse_dbStruct(structFile, stPath) #dataset에 대한 파일 읽기

        self.images = [os.path.join(dbPath, dbIm) for dbIm in self.dbStruct.dbImage]
        if not onlyDB:
            self.images += [os.path.join(qPath, qIm) for qIm in self.dbStruct.qImage]

        self.whichSet = self.dbStruct.whichSet  #train, test, val 중 하나
        self.dataset = self.dbStruct.dataset   # pittsburgh250k, 30k 등

        self.positives = None   #현재는 없음
        self.distances = None   #현재는 없음

    def __len__(self):
            return len(self.images)

    def __getitem__(self, index):
        img = Image.open(self.images[index])  #dataset의 이미지를 불러와 출력

        if self.input_transform:
            img = self.input_transform(img)  #tensor로 변환한다. 
        return img, index

    def getPositive(self):   #학습에선 사용하지 않음. 이후 Test/Evaluation에서 GT추출용으로 사용 
        
        #Data의 숫자가 크지 않아 sklearn으로 아직까지 가능할 듯.         
        if  self.positives is None:
            knn = NearestNeighbors(n_jobs=-1)
            knn.fit(self.dbStruct.utmDb)

            self.distances, self.positives = knn.radius_neighbors(self.dbStruct.utmQ,
                    radius=self.dbStruct.posDistThr)

        return self.positives

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]


In [51]:
whole_train_set = WholeDatset(
    dbPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_train.mat',
    transform=input_transform(),
    onlyDB=False
    )


whole_val_set = WholeDatset(
    dbPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_val.mat',
    transform=input_transform(),
    onlyDB=False
    )

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]
10
matStruct[0] :val
matStruct[1] :[array(['000/000000_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585200.91088968 585200.91088968 585200.91088968 ... 584439.93767933
 584439.93767933 584439.93767933]
matStruct[3] :[array(['000/000015_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585001.41335051 585001.41335051 585001.41335051 ... 584534.69627173
 584534.69627173 584534.69627173]
matStruct[5] :[10000]
matStruct[6] :[7608]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]


In [52]:
from torch.utils.data import DataLoader, SubsetRandomSampler

def getCluster(mymodel,datasets,K=64):

    #Parameter Initialize
    #테스트용 코드 작성과는 다르게, Ref.코드에서는 이미지 100개에서, 5만개의 Desciptor를 샘플링한다. 
    #따라서, 데이터셋에서 100개의 이미지를 랜덤샘플하고, 여기서 각각 500개의 descriptor를 추출해서 Clustering을 진행한다. 
    #https://m.blog.naver.com/kwangrok21/222412219800 SubsetRandomSampler 사용법법

    from math import ceil
    from torch.utils.data import SubsetRandomSampler
    import numpy as np
    from sklearn.cluster import KMeans
    

    nDesrciptor = 50000
    nPerImages = 100
    nIm = ceil(nDesrciptor / nPerImages)
    sampler = SubsetRandomSampler(np.random.choice(len(datasets), nIm, replace=False))

    data_loader = DataLoader(datasets, sampler=sampler)

    mymodel.eval()
    desc_list = []

    print(data_loader)
    count = 0 
    with torch.no_grad():
        for batch in data_loader:      #data_loader는 DataLoader object이지만, CNN입력은 tensor(B,C,H,W)이어야 하므로, loop로 돌려야 한다. 
            if isinstance(batch, (list, tuple)):
                x = batch[0]          # 보통 image, 지금 이쪽으로 들어온다. 
            else:
                x = batch
            count = count + 1
            
            print(f"x.type is {x.dtype} and {count}, batch shape : {batch[0].shape}")

            x = x.to(device)
            feat = mymodel(x)
            B,C,H,W = feat.shape
            feat = feat.permute(0,2,3,1).reshape(-1,C)

            idx = np.random.choice(feat.shape[0], nPerImages, replace=False)
            feat = feat[idx]

            desc_list.append(feat.cpu().numpy())

    X_np = np.concatenate(desc_list, axis=0)

    print("clustering")
    kmeans = KMeans(n_clusters=K, random_state=0, n_init=10)
    kmeans.fit(X_np)

    centroids = kmeans.cluster_centers_

    return centroids
    
    
    # 이건 getCluster사용하기 전에 먼저 할 것. 
    # x = x.to(device)

In [53]:
class VGG16Feature(nn.Module):
    def __init__(self):
        super().__init__()
        
        #encoder = models.vgg16(pretrained=True) 버전이 바뀌면서 워닝이 뜬다.
        encoder = models.vgg16(weights="VGG16_Weights.IMAGENET1K_FEATURES")
        # capture only feature part and remove last relu and maxpool
        layers = list(encoder.features.children())[:-2]
        
        self.encoder = nn.Sequential(*layers)
        self.encoder_dim = 512

        for p in self.encoder.parameters():
            p.requires_grad = False
    
    def forward(self, x):
        x = self.encoder(x)
        return x
#Class 끝 

#데이터 준비

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
])


In [54]:
#cuda 확인
if torch.cuda.is_available() :
    device = 'cuda'
else :
    device = 'cpu' 


In [55]:
model = VGG16Feature().to(device)
centroids = getCluster(model, whole_train_set)    

x.type is torch.float32 and 1, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 2, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 3, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 4, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 5, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 6, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 7, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 8, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 9, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 10, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 11, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 12, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 13, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 14, ba

In [56]:

class NetVLAD(nn.Module):
    #VLAD Layer
    def __init__(self, num_clusters = 64, dim = 128,normalize_input = True):
    
    #        Args:
    # num_clusters : int
    #     The number of clusters
    # dim : int
    #     Dimension of descriptors
    # alpha : float
    #     Parameter of initialization. Larger value is harder assignment.
    # normalize_input : bool
    #     If true, descriptor-wise L2 normalization is applied to input.

        super(NetVLAD, self).__init__()
        self.num_clusters = num_clusters  #cluster개수를 정의해야 vlad vector를 정의할 수 있다. 
        self.dim = dim   #ref코드는 128로 초기화했는데, 만약 VGG-16을 쓴다면 512를 써야 한다. 
        self.alpha = 0  #soft-assignment를 위한 alpha값. scratch 구현에선 2~10정도로 임의 설정했으나, 이제 이걸 학습해나가야 한다. 
        
        self.normalize_input = normalize_input  #이게 필요한가..? 정규화 여부를 저장한다. bool.
        self.conv = nn.Conv2d(dim, num_clusters, kernel_size=(1, 1), bias=False)  #assign용 연산. 아마도 alpha값과 centroid의 벡터곱 등에 쓴다. 
        self.centroids = nn.Parameter(torch.rand(num_clusters, dim))
        #자리 만들기. nn.Parameter 함수를 써서 학습파라미터로 선언한다. none으로 설정하면 네트워크 설정하면서 optimizer를 붙일수 없다. 
        #그래서 뭐라도 넣어놔야 한다. 


    def init_params(self, centroids, descriptors):
        #실제로 cluster의 centroid와 descritor를 받는 부분
        #이 함수를 통해 c_k, alpha, conv.weight와 conv.bias를 통해 assignment score(z_k)를 계산한다. 
        #즉 soft-assignment 관련 항목을 초기화한다. 

        #해야 할것 
        # 1. centroid c_k를 학습할 수 있도록 파라미터로 등록
        # 2. assignment score z_k를 계산하는 self.conv(x)의 weight를 셋팅한다. 
            
            
            #타입 맞추기
            device = descriptors.device
            dtype = descriptors.dtype


            #getCluster를 연산하면 numpy형태로 반환받는다. 그걸 텐서로 바꾼다.
            #기존 scratch에서는 torch.tensor를 사용했고, 이번에는 torch.as_tensor를 사용한다. 
            # 참고자료 https://jh-bk.tistory.com/46
            
            centroids_t = torch.as_tensor(centroids, dtype=dtype, device=device)   # (K=64, dim=512)

            desc_norm = F.normalize(descriptors, p=2, dim=1)       # (M=B*H*W, C)
            cent_norm = F.normalize(centroids_t, p=2, dim=1)       # (K, C)

            # W^T = 2 * alpha * c_k.t() * x_i  

            dots = torch.matmul(cent_norm, desc_norm.t())          
            # (K, M)형태로 출력하기 위해 편의상 transpose의 위치가 바뀐다. 
            # 이는 ref. 코드에서도 동일하다. dots = np.dot(clstsAssign, traindescs.T)
            dots, _ = torch.sort(dots, dim=0, descending=True)

            self.alpha = (-torch.log(torch.tensor(0.01, device=device, dtype=dtype))
                        / torch.mean(dots[0, :] - dots[1, :])).item()

            self.centroids = nn.Parameter(centroids_t)

            self.conv.weight = nn.Parameter(
                (self.alpha * cent_norm).unsqueeze(-1).unsqueeze(-1)   # (K, C, 1, 1)
            )
            self.conv.bias = None

    def forward(self, x):
        N, C = x.shape[:2]

        if self.normalize_input:
            x = F.normalize(x, p=2, dim=1)   # (N, C, H, W)

        soft_assign = self.conv(x)                               # (N, K, H, W)
        soft_assign = soft_assign.view(N, self.num_clusters, -1) # (N, K, HW)
        soft_assign = F.softmax(soft_assign, dim=1)              # (N, K, HW)

        x_flatten = x.view(N, C, -1)                             # (N, C, HW)

        vlad = torch.zeros(
            N, self.num_clusters, C,
            dtype=x.dtype,
            device=x.device
        )                                                        # (N, K, C)

        for k in range(self.num_clusters):
            centroid = self.centroids[k].view(1, C, 1)           # (1, C, 1)
            residual = x_flatten - centroid                      # (N, C, HW)

            assign_weight = soft_assign[:, k, :].view(N, 1, -1)  # (N, 1, HW)
            residual = residual * assign_weight                  # (N, C, HW)

            vlad[:, k, :] = residual.sum(dim=2)                  # (N, C)

        vlad = F.normalize(vlad, p=2, dim=2)                     # (N, K, C)
        vlad = vlad.view(N, -1)                                  # (N, K*C)
        vlad = F.normalize(vlad, p=2, dim=1)                     # (N, K*C)

        return vlad

In [57]:
encoder = VGG16Feature()
pool = NetVLAD(num_clusters=64, dim=512)


model = nn.Module()
model.add_module('encoder', encoder)
model.add_module('pool', pool)

model.to(device)

print(model)

Module(
  (encoder): VGG16Feature(
    (encoder): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=True)
   

# Training 을 위한 구성. 

## Training Dataset
  - Dataset형태로 선언(Class)
  - __init__(self)  
     - 데이터 선언: 기본데이터/mat파일로부터 데이터 파싱
     - Positive 분류, Negative 분류
  - __getitem__(self)  
    - query 이미지
    - positive 이미지
    - negative 이미지

  - __getlen__(self)  
    - query 이미지가 곧 길이가 된다다. 



In [58]:
class QueryDataset(data.Dataset):
    def __init__(self, rootPath, stPath, qPath, structFile, nNegSample = 1000, nNeg=10, margin = 0.1, input_transform = None):
        super().__init__()

        self.input_transform = input_transform
        self.margin = margin

        self.rootPath = rootPath
        self.stPath = stPath
        self.qPath = qPath
        self.structFile = structFile

        self.dbStruct = parse_dbStruct(structFile, stPath) #dataset에 대한 파일 읽기

        self.nNegSample = nNegSample # number of negatives to randomly sample
        self.nNeg = nNeg # number of negatives used for training

        #Test/Eval 시 WholeDataset 만으로도 수행 가능하도록 images도 선언함. (query + db) or(only db)
        self.db_images = [os.path.join(self.rootPath, dbIm) for dbIm in self.dbStruct.dbImage]
        self.q_images = [os.path.join(self.qPath, qIm) for qIm in self.dbStruct.qImage]

        self.positives = None   #현재는 없음
        self.distances = None   #현재는 없음


        #mat data load and parsing
        self.whichSet = self.dbStruct.whichSet              #train
        self.dataset = self.dbStruct.dataset                #pittsburch250k


        #Positive 계산을 위한 사전작업
        knn = NearestNeighbors(n_jobs=-1)
        knn.fit(self.dbStruct.utmDb)

        #거리 25m 이내의 positive 후보군들을 검색한다. 
        #기본 설정은 민코프스키 로 되어있으므로, 자동으로 유클리디안 거리가 나온다. 
        #여기서 의문은, trivial positive를 어떻게 제외하지 하는 부분이다. 

        PositiveDistance,PositiveIndex = knn.radius_neighbors(
            self.dbStruct.utmQ,
            radius=self.dbStruct.posDistThr,
            return_distance=True
            )

        nontrivial_positives = []

        for i in range(len(PositiveIndex)):
            fDist = PositiveDistance[i]
            nIndex = PositiveIndex[i]
            fDistSq = fDist ** 2
            mask = fDistSq > self.dbStruct.nonTrivPosDistSqThr
            #np masking 방법 https://m.blog.naver.com/baek2sm/221844619151
            nontrivial_positives.append(nIndex[mask])

        self.pos_within_Thr = PositiveIndex  #trivial 도 포함
        self.nontrivial_pos = nontrivial_positives 

        #예외처리를 위한 부분. 만약 nontrivial possitive가 없다면 해당 쿼리는 제외한다. 
        self.queries = np.where(np.array([len(x) for x in self.nontrivial_pos])>0)[0]

        # Negative 후보 만들기 
        self.potential_negatives = []        

        # Index로 연산하기 위해 전체 Index를 하나 만든다. 
        self.numDbIndex = np.arange(self.dbStruct.numDb)

        #전체 dbImage 배열에서 Potential Positve를 뺀 나머지 배열을 만든다. 
        for pos in self.pos_within_Thr:
            self.potential_negatives.append(
                                            np.setdiff1d(
                                                         self.numDbIndex, pos, 
                                                         assume_unique=True
                                                         )
                                            )       

    def getPositive(self):   #학습에선 사용하지 않음. 이후 Test/Evaluation에서 GT추출용으로 사용 
        return self.pos_within_Thr

    def getNegative(self):
        return self.potential_negatives

    def getNontrivialPositive(self):
        return self.nontrivial_pos
    
    def getValidQueries(self):
        return self.queries

    def __getitem__(self, idx):
        q_idx = self.queries[idx]

        pos_idx = np.random.choice(self.nontrivial_pos[q_idx])
        neg_idx = np.random.choice(self.potential_negatives[q_idx])

        q_img = Image.open(self.q_images[q_idx]).convert("RGB")
        p_img = Image.open(self.db_images[pos_idx]).convert("RGB")
        n_img = Image.open(self.db_images[neg_idx]).convert("RGB")

        if self.input_transform is not None:
            q_img = self.input_transform(q_img)
            p_img = self.input_transform(p_img)
            n_img = self.input_transform(n_img)

        return q_img, p_img, n_img

    def __len__(self):
        return len(self.queries)


query_train_set = QueryDataset(
    rootPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_train.mat',
    input_transform = input_transform()
    )

print(query_train_set.getPositive())

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]
[array([  0,  17,  18,  19,  20,  21,  22,  23,  16,  15,   1,   2,   3,
          4,   5,   6,   7,   8,   9,  10,  11,  12,  35,  34,  33,  24,
         25,  26,  27,  28,  29,  30,  32,  31,  13,  14, 124, 120, 121,
        122, 123, 126, 125, 135, 127, 128, 129, 130, 134, 131, 132, 133,
        137, 136, 140, 138, 139, 141, 143, 142,  72,  83,  82,  81,  80,
         79,  78,  77,  76,  75,  74,  84,  73,  87,  86,  85,  88,  89,
         90,  91,  92,  95,  94,  93,  41,  36,  42,  43,  44,  45, 

In [59]:
loader = DataLoader(query_train_set, batch_size=2, shuffle=True)

q, p, n = next(iter(loader))

print(q.shape)
print(p.shape)
print(n.shape)

torch.Size([2, 3, 480, 640])
torch.Size([2, 3, 480, 640])
torch.Size([2, 3, 480, 640])


In [60]:
class EmbedNet(nn.Module):
    def __init__(self, encoder, pool):
        super().__init__()
        self.encoder = encoder
        self.pool = pool

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        return x


encoder = VGG16Feature()
pool = NetVLAD(num_clusters=64, dim=512)

model = EmbedNet(encoder, pool).to(device)

In [61]:
train_loader = DataLoader(
    query_train_set,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

In [62]:
criterion = nn.TripletMarginLoss(margin=0.1, p=2)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

# TensorBoard + Train/Val

아래 셀은 다음을 수행한다.

1. epoch마다 `train loss` 기록  
2. epoch마다 validation set에서 `Recall@1/5/10` 계산  
3. TensorBoard에 scalar 기록  
4. best `Recall@1` 모델 저장

> 주의: validation 전체 descriptor 추출은 시간이 오래 걸릴 수 있다.


In [63]:
from torch.utils.tensorboard import SummaryWriter
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader


In [68]:
def evaluate_recall(model, whole_set, device, batch_size=4, num_workers=0, topk=(1, 5, 10)):
    model.eval()

    loader = DataLoader(
        whole_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    all_desc = []


    with torch.no_grad():
        for images, indices in loader:
            images = images.to(device)
            desc = model.pool(model.encoder(images))
            all_desc.append(desc.cpu())  #CPU 에서 연산을 위한 이동. sklearn이 CPU에서 돌아간다. 만약 FAISS라면 GPU연산이 가능하다. 
            all_indices.append(indices)

    all_desc = torch.cat(all_desc, dim=0)

    n_db = whole_set.dbStruct.numDb
    n_q = whole_set.dbStruct.numQ

    db_desc = all_desc[:n_db].numpy()
    q_desc = all_desc[n_db:n_db + n_q].numpy()

    knn = NearestNeighbors(
        n_neighbors=max(topk),
        metric='euclidean',
        n_jobs=-1
    )
    knn.fit(db_desc)
    distances, indices = knn.kneighbors(q_desc)

    utmDb = whole_set.dbStruct.utmDb
    utmQ = whole_set.dbStruct.utmQ
    posDistThr = whole_set.dbStruct.posDistThr

    knn_gt = NearestNeighbors(n_jobs=-1)
    knn_gt.fit(utmDb)
    positives = knn_gt.radius_neighbors(
        utmQ,
        radius=posDistThr,
        return_distance=False
    )

    recalls = {}
    for k in topk:
        correct = 0
        for i in range(len(q_desc)):
            pred = indices[i, :k]
            gt = positives[i]
            if np.intersect1d(pred, gt).size > 0:
                correct += 1
        recalls[k] = correct / len(q_desc)

    return recalls


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, writer=None, epoch=0):
    model.train()
    running_loss = 0.0
    
    #Batch loop
    for batch_idx, (q, p, n) in enumerate(loader):
        q = q.to(device)
        p = p.to(device)
        n = n.to(device)

        optimizer.zero_grad()

        q_desc = model(q)
        p_desc = model(p)
        n_desc = model(n)

        loss = criterion(q_desc, p_desc, n_desc)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()


        #TensorBoard용 코드
        global_step = epoch * len(loader) + batch_idx
        if writer is not None:
            writer.add_scalar("train/loss_iter", loss.item(), global_step)

        if batch_idx % 10 == 0:
            print(f"epoch {epoch+1} | batch {batch_idx}/{len(loader)} | loss = {loss.item():.4f}")

    avg_loss = running_loss / len(loader)

    if writer is not None:
        writer.add_scalar("train/loss_epoch", avg_loss, epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)

    return avg_loss


In [ ]:
# TensorBoard 실행 전에 log_dir를 먼저 만든다.
log_dir = "runs/netvlad_trainval"
writer = SummaryWriter(log_dir=log_dir)

In [ ]:


num_epochs = 3
val_every = 1
best_r1 = -1.0
best_ckpt_path = "best_netvlad_model.pth"

for epoch in range(num_epochs):
    avg_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        writer=writer,
        epoch=epoch
    )

    print(f"[Epoch {epoch+1}] train loss = {avg_loss:.4f}")

    if (epoch + 1) % val_every == 0:
        recalls = evaluate_recall(
            model=model,
            whole_set=whole_val_set,
            device=device,
            batch_size=4,
            num_workers=0,
            topk=(1, 5, 10)
        )

        writer.add_scalar("val/Recall@1", recalls[1], epoch)
        writer.add_scalar("val/Recall@5", recalls[5], epoch)
        writer.add_scalar("val/Recall@10", recalls[10], epoch)

        print(
            f"[Epoch {epoch+1}] "
            f"R@1={recalls[1]:.4f}, "
            f"R@5={recalls[5]:.4f}, "
            f"R@10={recalls[10]:.4f}"
        )

        if recalls[1] > best_r1:
            best_r1 = recalls[1]
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"best model updated -> {best_ckpt_path} (R@1={best_r1:.4f})")

writer.close()
print("training + validation finished")
print(f"log_dir: {log_dir}")
print(f"best Recall@1: {best_r1:.4f}")


epoch 1 | batch 0/1854 | loss = 0.0991
epoch 1 | batch 10/1854 | loss = 0.0965
epoch 1 | batch 20/1854 | loss = 0.0991
epoch 1 | batch 30/1854 | loss = 0.1006
epoch 1 | batch 40/1854 | loss = 0.1000
epoch 1 | batch 50/1854 | loss = 0.0991
epoch 1 | batch 60/1854 | loss = 0.0918
epoch 1 | batch 70/1854 | loss = 0.1005
epoch 1 | batch 80/1854 | loss = 0.0957
epoch 1 | batch 90/1854 | loss = 0.0991
epoch 1 | batch 100/1854 | loss = 0.0943
epoch 1 | batch 110/1854 | loss = 0.0990
epoch 1 | batch 120/1854 | loss = 0.0993
epoch 1 | batch 130/1854 | loss = 0.0960
epoch 1 | batch 140/1854 | loss = 0.1010
epoch 1 | batch 150/1854 | loss = 0.0948
epoch 1 | batch 160/1854 | loss = 0.0997
epoch 1 | batch 170/1854 | loss = 0.1000
epoch 1 | batch 180/1854 | loss = 0.0939
epoch 1 | batch 190/1854 | loss = 0.0992
epoch 1 | batch 200/1854 | loss = 0.0953
epoch 1 | batch 210/1854 | loss = 0.0979
epoch 1 | batch 220/1854 | loss = 0.0999
epoch 1 | batch 230/1854 | loss = 0.1004
epoch 1 | batch 240/1854 | 

TensorBoard는 아래 둘 중 하나로 확인할 수 있다.

1. VS Code Command Palette → `TensorBoard: Start` → `runs` 폴더 선택  
2. 노트북 셀에서 아래 magic 실행


In [67]:
%load_ext tensorboard
%tensorboard --logdir runs
